<a href="https://colab.research.google.com/github/nabillahumi/S5-Mesin_Learning/blob/main/Jobsheet3%20PBL/Tugas%20Praktikum%20(JS3)%20PBL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Pisahkan antara variabel yang dapat digunakan dan variabel yang tidak dapat digunakan.

In [2]:
import pandas as pd

# 1. Load dan Gabungkan Dataset
df_cafes = pd.read_csv('cafes.csv')
df_reviews = pd.read_csv('reviews.csv')

df_reviews['text_len'] = df_reviews['text'].fillna('').str.len()
agg_reviews = df_reviews.groupby('cafe_id').agg(
    avg_review_rating=('rating', 'mean'),
    total_likes=('likes', 'sum'),
    extracted_review_count=('review_id', 'count')
).reset_index()

df = pd.merge(df_cafes, agg_reviews, on='cafe_id', how='left')

# 1. Menentukan kolom yang tidak digunakan (drop) dan kolom yang digunakan
cols_to_drop = ['cafe_id', 'name', 'address', 'phone', 'website', 'scraped_at']
df_filtered = df.drop(columns=cols_to_drop)

# Tampilkan 5 data teratas dari hasil pemilihan variabel
df_filtered.head()

,latitude,longitude,rating,review_count,categories,source_query,avg_review_rating,total_likes,extracted_review_count
0,-7.972556,112.622708,4.7,528,['Kedai Kopi'],cafe Kota Malang,4.400000,0.0,30.0
1,-7.963825,112.627146,4.7,141,['Kafe'],cafe Kota Malang,4.866667,0.0,30.0
2,-7.972555,112.618455,4.6,1772,['Kafe'],cafe Kota Malang,4.217391,0.0,23.0
3,-7.967561,112.633346,4.3,701,['Kedai Kopi'],cafe Kota Malang,4.133333,0.0,30.0
4,-7.976481,112.634028,4.3,686,['Kafe'],cafe Kota Malang,3.666667,0.0,30.0


2. Lakukan proses encoding pada kolom Target / Kategori.

In [3]:
from sklearn.preprocessing import LabelEncoder

# Membuat objek LabelEncoder
le = LabelEncoder()

# Mendeteksi kolom kategorikal yang tersisa pada df_filtered
cat_cols = df_filtered.select_dtypes(include=['object']).columns.tolist()

# Melakukan proses encoding untuk setiap kolom kategorikal
for col in cat_cols:
    df_filtered[col] = le.fit_transform(df_filtered[col].astype(str))

# Menampilkan 5 data teratas setelah encoding kolom kategori
df_filtered[['categories', 'source_query']].head()

,categories,source_query
0,2,0
1,1,0
2,1,0
3,2,0
4,1,0


3. Lakukan proses standarisasi pada semua kolom yang memiliki nilai numerik.

In [4]:
from sklearn.preprocessing import StandardScaler

# Membuat objek StandardScaler
scaler = StandardScaler()

# Mendeteksi semua kolom yang bertipe numerik
num_cols = df_filtered.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()

# Melakukan standardisasi pada seluruh kolom numerik
df_filtered[num_cols] = scaler.fit_transform(df_filtered[num_cols])

# Menampilkan 5 baris teratas setelah standardisasi
df_filtered.head()

,latitude,longitude,rating,review_count,categories,source_query,avg_review_rating,total_likes,extracted_review_count
0,-0.782578,-0.671833,0.239675,-0.304432,0.454369,-1.085292,-0.417755,-0.366465,0.62670
1,0.029950,-0.066581,0.239675,-0.622469,-1.419905,-1.085292,0.740478,-0.366465,0.62670
2,-0.782475,-1.251843,-0.325274,0.717889,-1.419905,-1.085292,-0.870976,-0.366465,-0.07223
3,-0.317752,0.778786,-2.020122,-0.162260,0.454369,-1.085292,-1.079602,-0.366465,0.62670
4,-1.147860,0.871843,-2.020122,-0.174587,-1.419905,-1.085292,-2.237835,-0.366465,0.62670


4. Lakukan proses stratified split data untuk membuat data latih dan data uji dengan rasio 80:20.

In [5]:
from sklearn.model_selection import train_test_split

# Menentukan kolom target biner untuk stratifikasi (misal: berdasarkan rating >= 4.5)
df_filtered['target_bin'] = (df['rating'] >= 4.5).astype(int)

# Memisahkan fitur (X) dan target (y)
X = df_filtered.drop(columns=['target_bin'])
y = df_filtered['target_bin']

# Melakukan stratified split dengan rasio 80:20 (test_size=0.2)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Menampilkan informasi ukuran dan distribusi kelas hasil split
print(f"Jumlah data train: {X_train.shape[0]}")
print(f"Jumlah data test: {X_test.shape[0]}")
print("\nDistribusi label pada y_train:")
print(y_train.value_counts())
print("\nDistribusi label pada y_test:")
print(y_test.value_counts())

Jumlah data train: 79
Jumlah data test: 20

Distribusi label pada y_train:
target_bin
1    68
0    11
Name: count, dtype: int64

Distribusi label pada y_test:
target_bin
1    17
0     3
Name: count, dtype: int64
